# Manchester City — Reusable Match Forecaster

**Use this same notebook for every future Manchester City Premier League match.** No opponent-specific ZIP or model retraining is required.

Before each forecast:
1. Add all newly completed league results to `epl_results.csv` (keeping the five-column format).
2. Change the three `NEXT_...` values in Step 1.
3. Run the notebook from top to bottom **before kickoff**.
4. Keep `season_state.json`: it saves recorded results **and** forecasts for later evaluation.

**Leakage rule:** This Version 1 uses **dates, not kickoff times**. For a match on date D, the importer uses only results with dates **strictly earlier** than D. Never issue a forecast after kickoff and then label it prospective. Any same-day results should only be entered after the forecast is saved.

The CSV is a separate, editable match-result log. Exact duplicate match records are ignored when you import it again. Imported results update Bayesian team strengths; frozen model coefficients do not change.


## Step 1 — Set your next fixture

The values below are an **example for 11 October 2026**. For every new forecast, edit these three lines. `NEXT_VENUE` is always from **Manchester City's perspective**. Team names must match `online.strength_table()` exactly.


In [1]:
NEXT_DATE = "2026-10-11"
NEXT_OPPONENT = "Liverpool"
NEXT_VENUE = "Away"    # "Home" or "Away", from Man City's perspective


## Step 2 — Load your saved model

Open the `forecaster` directory as your VS Code working directory. `season_state.json` and `city_online_forecast.py` must be in this same directory. Install requirements once with `python -m pip install -r requirements.txt`.


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from city_online_forecast import CitySeasonForecaster

STATE_FILE = Path("season_state.json")
if not STATE_FILE.exists():
    raise FileNotFoundError("Missing season_state.json. Open the forecaster folder first.")

online = CitySeasonForecaster.load(STATE_FILE)
display(online.summary())
print("Team names expected by model:")
print(", ".join(online.strength_table()["Team"].tolist()))


,Target_Season,Selected_Model,N_Teams,N_Results,N_City_Results,Last_Result_Date,N_Saved_Forecasts
0,2026/27,Strength_Home,20,5,5,2026-09-20,0


Team names expected by model:
Man City, Arsenal, Liverpool, Man United, Chelsea, Aston Villa, Newcastle, Brighton, Bournemouth, Brentford, Nott'm Forest, Fulham, Everton, Tottenham, Crystal Palace, Sunderland, Leeds, Coventry, Ipswich, Hull


## Step 3 — Import all results before your fixture

Maintain **one** file called `epl_results.csv` with these headers:

`Date,HomeTeam,AwayTeam,FTHG,FTAG`

Example format: `2026-08-23,Man City,Bournemouth,2,1` (a match already included in the initial state).

Add each new finished Premier League match as another row. You can keep old rows: exact duplicates are skipped. Do not include scheduled matches, blanks, cup matches, or unknown clubs. The importer selects rows dated strictly before `NEXT_DATE`.


In [3]:
CSV_FILE = Path("epl_results.csv")
REQUIRED_COLS = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]

if not CSV_FILE.exists():
    print("No epl_results.csv found. Continuing with results already in season_state.json.")
else:
    all_results = pd.read_csv(CSV_FILE, dtype={"Date": str})
    missing = [col for col in REQUIRED_COLS if col not in all_results.columns]
    if missing:
        raise ValueError(f"Missing columns in epl_results.csv: {missing}")

    date_values = pd.to_datetime(all_results["Date"], format="%Y-%m-%d", errors="raise")
    all_results = all_results.copy()
    all_results["Date"] = date_values.dt.strftime("%Y-%m-%d")
    eligible = all_results.loc[all_results["Date"] < NEXT_DATE, REQUIRED_COLS].copy()
    print(f"Rows before {NEXT_DATE}: {len(eligible)}")

    if not eligible.empty:
        update = online.record_results(eligible)
        display(update)
    else:
        print("No earlier-date rows to import. Existing saved results are preserved.")


Rows before 2026-10-11: 56


,Target_Season,Selected_Model,N_Teams,N_Results,N_City_Results,Last_Result_Date,N_Saved_Forecasts,Added_Results
0,2026/27,Strength_Home,20,56,5,2026-10-10,0,51


## Step 4 — Issue and save the forecast

**Run this only before kickoff.** A successful call to `online.predict()` immediately saves a snapshot of the forecast in `season_state.json`. It displays probabilities from Manchester City's perspective (including away matches). Do not change your model or reissue alternative forecasts based on seeing the answer.


In [4]:
prediction = online.predict(
    opponent=NEXT_OPPONENT,
    venue=NEXT_VENUE,
    date=NEXT_DATE,
)

print("MATCH")
display(prediction["Match"].round(4))
print("WIN / DRAW / LOSS PROBABILITIES (%)")
display(prediction["Outcome_Probabilities_pct"].round(2))
print("TOP 5 EXACT SCORES (%) — City goals first")
display(prediction["Top5"].round(2))
print("0–5 SCORE GRID (%) — rows: City goals; columns: opponent goals")
display((prediction["Grid_0_5"] * 100).round(2))


MATCH


,Forecast_ID,Date,Venue,Opponent_Key,Model,S_City,S_Opp,Delta_S,Lambda_City,Lambda_Opp,N_Results_Used,Last_Result_Date
0,f400605faa472b22,2026-10-11,Away,Liverpool,Strength_Home,3.1616,1.6767,1.4848,1.5105,1.2546,56,2026-10-10


WIN / DRAW / LOSS PROBABILITIES (%)


,P_Win,P_Draw,P_Loss,Top5_Probability_Mass,P_Outside_0_5
0,43.2,25.23,31.57,45.84,0.65


TOP 5 EXACT SCORES (%) — City goals first


,Score_City_Opp,Probability_pct
Rank,,
1,1-1,11.93
2,1-0,9.51
3,2-1,9.01
4,0-1,7.90
5,1-2,7.49


0–5 SCORE GRID (%) — rows: City goals; columns: opponent goals


Opponent goals,0,1,2,3,4,5
City goals,,,,,,
0,6.30,7.90,4.96,2.07,0.65,0.16
1,9.51,11.93,7.49,3.13,0.98,0.25
2,7.18,9.01,5.65,2.36,0.74,0.19
3,3.62,4.54,2.85,1.19,0.37,0.09
4,1.37,1.71,1.07,0.45,0.14,0.04
5,0.41,0.52,0.32,0.14,0.04,0.01


## Step 5 — Review forecasts later

After the fixture ends, add its actual result (and other completed EPL matches) to `epl_results.csv`. Importing it before the **next** forecast updates team strengths and lets the report compare past forecasts with reality. This cell does not create forecasts.


In [5]:
print("CURRENT MODEL STATUS")
display(online.summary())
print("SAVED FORECAST HISTORY")
display(online.forecast_history())
print("CITY SEASON REPORT")
display(online.season_report(replay_missing=False))


CURRENT MODEL STATUS


,Target_Season,Selected_Model,N_Teams,N_Results,N_City_Results,Last_Result_Date,N_Saved_Forecasts
0,2026/27,Strength_Home,20,56,5,2026-10-10,1


SAVED FORECAST HISTORY


,Forecast_ID,Date,Venue,Opponent_Key,Model,S_City,S_Opp,Delta_S,Lambda_City,Lambda_Opp,...,Top1,Top2,Top3,Top4,Top5,P_Win,P_Draw,P_Loss,Top5_Probability_Mass,P_Outside_0_5
0,f400605faa472b22,2026-10-11,Away,Liverpool,Strength_Home,3.161564,1.67673,1.484834,1.510451,1.254644,...,1-1 (11.93%),1-0 (9.51%),2-1 (9.01%),0-1 (7.90%),1-2 (7.49%),0.432043,0.252267,0.31569,0.458436,0.006469


CITY SEASON REPORT


,Date,Venue,Opponent,Actual_Score,Prediction_Source,Forecast_ID,Forecast_For_Date,N_Results_Used,Lambda_City,Lambda_Opp,Top1,Top2,Top3,Top4,Top5,Actual_Rank,Top5_Hit
0,2026-08-23,Home,Bournemouth,2-1,No_saved_forecast,None,None,NaN,NaN,NaN,None,None,None,None,None,None,None
1,2026-08-28,Away,Crystal Palace,4-1,No_saved_forecast,None,None,NaN,NaN,NaN,None,None,None,None,None,None,None
2,2026-09-05,Home,Coventry,1-0,No_saved_forecast,None,None,NaN,NaN,NaN,None,None,None,None,None,None,None
3,2026-09-13,Away,Man United,1-0,No_saved_forecast,None,None,NaN,NaN,NaN,None,None,None,None,None,None,None
4,2026-09-20,Home,Sunderland,5-3,No_saved_forecast,None,None,NaN,NaN,NaN,None,None,None,None,None,None,None
5,2026-10-11,Away,Liverpool,Pending,Saved_same_day_time_unverified,f400605faa472b22,2026-10-11,56.0,1.510451,1.254644,1-1 (11.93%),1-0 (9.51%),2-1 (9.01%),0-1 (7.90%),1-2 (7.49%),None,None
